# Variable Construction

This notebook constructs the derived variables used in the subsequent descriptive and statistical analyses.

Main steps:
- Rename and standardize mechanization indicators
- Construct annual agricultural employment change variables
- Construct agricultural employment per 1,000 hectares
- Construct net migration indicators and migration rates
- Validate and export the final analytical dataset


In [8]:
import sys
!{sys.executable} -m pip install pandas numpy


[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
import numpy as np
import pandas as pd

In [35]:
df = pd.read_csv("../../data/processed/analysis_dataset_ins.csv")
df.head(2)

,an,judet,emig_masc_nr,emig_fem_nr,emigranti_total_nr,emig_15_64_nr,imig_masc_nr,imig_fem_nr,imigranti_total_nr,imigranti_15_64_nr,...,z_rezid_combine,indice_mecanizare_rezidual,schimbare_ocupati_agri_nr,schimbare_ocupati_agri_pct,schimbare_pondere_ocupati_agri,ocupati_agri_1000ha,migratie_neta_total_nr,rata_migratie_neta_total,migratie_neta_15_64_nr,rata_migratie_neta_15_64
0,2012,Alba,143,144,287,222,43,33,76,66,...,0.802190,0.325906,NaN,NaN,NaN,772.194749,-211,-0.619597,-156,-0.679031
1,2013,Alba,104,158,262,224,53,41,94,83,...,0.841744,0.412092,-2400.0,-4.848485,-1.451137,746.339608,-168,-0.496060,-141,-0.618020


In [11]:
# redenumire indici mecanizare 

df = df.rename(columns={
    "indice_mecanizare_vechi": "indice_mecanizare_densitate"
})

In [12]:
[col for col in df.columns if "indice_mecanizare" in col]

['indice_mecanizare_densitate', 'indice_mecanizare_rezidual']

In [13]:
# variabile schimbare forta de munca

df = df.drop(columns={"schimbare_pondere_ocupati_agri"})

In [14]:
df = df.sort_values(["judet", "an"]).copy()

df["schimbare_ocupati_agri_nr"] = (
    df.groupby("judet")["ocupati_agri_nr"].diff()
)

df["schimbare_ocupati_agri_pct"] = (
    df.groupby("judet")["ocupati_agri_nr"].pct_change() * 100
)

df["schimbare_pondere_ocupati_agri"] = (
    df.groupby("judet")["pondere_ocupati_agri"].diff()
)

In [15]:
df[[
    "judet", "an",
    "ocupati_agri_nr",
    "schimbare_ocupati_agri_nr",
    "schimbare_ocupati_agri_pct",
    "pondere_ocupati_agri",
    "schimbare_pondere_ocupati_agri"
]].head(15)

,judet,an,ocupati_agri_nr,schimbare_ocupati_agri_nr,schimbare_ocupati_agri_pct,pondere_ocupati_agri,schimbare_pondere_ocupati_agri
0,Alba,2012,49500,NaN,NaN,30.293758,NaN
1,Alba,2013,47100,-2400.0,-4.848485,28.842621,-1.451137
2,Alba,2014,45300,-1800.0,-3.821656,28.383459,-0.459162
3,Alba,2015,39800,-5500.0,-12.141280,25.000000,-3.383459
4,Alba,2016,34400,-5400.0,-13.567839,21.513446,-3.486554
5,Alba,2017,35000,600.0,1.744186,21.591610,0.078164
6,Alba,2018,35100,100.0,0.285714,21.600000,0.008390
7,Alba,2019,34800,-300.0,-0.854701,21.310472,-0.289528
8,Alba,2020,33500,-1300.0,-3.735632,20.577396,-0.733076
9,Alba,2021,17200,-16300.0,-48.656716,11.805079,-8.772317


In [16]:
df[[
    "schimbare_ocupati_agri_nr",
    "schimbare_ocupati_agri_pct",
    "schimbare_pondere_ocupati_agri"
]].isna().sum()

schimbare_ocupati_agri_nr         41
schimbare_ocupati_agri_pct        41
schimbare_pondere_ocupati_agri    41
dtype: int64

In [17]:
[col for col in df.columns if "sup" in col.lower()]

['sup_grau_ha',
 'sup_orz_orzoaica_ha',
 'sup_porumb_boabe_ha',
 'sup_floarea_soarelui_ha',
 'sup_rapita_ha',
 'sup_soia_boabe_ha',
 'sup_totala_cultivata_ha',
 'log_suprafata']

In [18]:
# variabile relative pentru forta de munca
df["ocupati_agri_1000ha"] = (
    df["ocupati_agri_nr"] / df["sup_totala_cultivata_ha"]
) * 1000

# numărul de persoane ocupate în agricultură, silvicultură și pescuit la 1000 ha suprafață cultivată

In [19]:
df[["ocupati_agri_nr", "sup_totala_cultivata_ha", "ocupati_agri_1000ha"]].describe().T

,count,mean,std,min,25%,50%,75%,max
ocupati_agri_nr,451.0,43302.217295,20009.888543,9100.00000,28150.000000,40700.000000,56200.000000,112300.000000
sup_totala_cultivata_ha,451.0,164187.168514,124399.791912,17077.00000,55475.000000,121925.000000,241820.500000,462162.000000
ocupati_agri_1000ha,451.0,494.920793,493.366014,48.08974,171.636472,369.194843,620.253984,3465.120526


In [20]:
df[[
    "judet", "an", "ocupati_agri_nr", 
    "sup_totala_cultivata_ha", "ocupati_agri_1000ha"
]].sort_values("ocupati_agri_1000ha", ascending=False).head(10)

,judet,an,ocupati_agri_nr,sup_totala_cultivata_ha,ocupati_agri_1000ha
275,Maramures,2012,75900,21904,3465.120526
276,Maramures,2013,71700,22677,3161.793888
277,Maramures,2014,69100,22721,3041.239382
278,Maramures,2015,59900,23118,2591.054589
220,Harghita,2012,44200,17077,2588.276629
280,Maramures,2017,51900,21391,2426.254032
221,Harghita,2013,41900,17928,2337.126283
282,Maramures,2019,51900,23390,2218.896965
281,Maramures,2018,52300,23975,2181.438999
222,Harghita,2014,40700,18805,2164.318001


In [21]:
#construiesc variabile migratie 


# migratie_neta totala
df["migratie_neta_total_nr"] = (
    df["imigranti_total_nr"] - df["emigranti_total_nr"]
)

In [22]:
# rata migratie neta total

df["rata_migratie_neta_total"] = (
    df["migratie_neta_total_nr"] / df["populatie_totala"]
) * 1000

In [23]:
# migratie neta 15-64

df["migratie_neta_15_64_nr"] = (
    df["imigranti_15_64_nr"] - df["emig_15_64_nr"]
)

In [24]:
# rata migratie neta 15-64

df["rata_migratie_neta_15_64"] = (
    df["migratie_neta_15_64_nr"] / df["populatie_15_64"]
) * 1000

In [25]:
df[[
    "judet", "an",
    "emigranti_total_nr", "imigranti_total_nr", 
    "migratie_neta_total_nr", "rata_migratie_neta_total",
    "emig_15_64_nr", "imigranti_15_64_nr",
    "migratie_neta_15_64_nr", "rata_migratie_neta_15_64"
]].head()

,judet,an,emigranti_total_nr,imigranti_total_nr,migratie_neta_total_nr,rata_migratie_neta_total,emig_15_64_nr,imigranti_15_64_nr,migratie_neta_15_64_nr,rata_migratie_neta_15_64
0,Alba,2012,287,76,-211,-0.619597,222,66,-156,-0.679031
1,Alba,2013,262,94,-168,-0.496060,224,83,-141,-0.618020
2,Alba,2014,161,94,-67,-0.198892,141,83,-58,-0.256297
3,Alba,2015,193,106,-87,-0.259857,175,88,-87,-0.388742
4,Alba,2016,288,119,-169,-0.508535,268,105,-163,-0.738904


In [26]:
df[[
    "judet", "an", "rata_migratie_neta_15_64",
    "imigranti_15_64_nr", "emig_15_64_nr", "populatie_15_64"
]].sort_values("rata_migratie_neta_15_64", ascending=False).head(10)

,judet,an,rata_migratie_neta_15_64,imigranti_15_64_nr,emig_15_64_nr,populatie_15_64
433,Vaslui,2019,43.393908,10776,711,231945
431,Vaslui,2018,40.253644,10088,604,235606
429,Vaslui,2017,35.734993,9002,435,239737
439,Vaslui,2022,30.983399,9105,1925,231737
423,Vaslui,2014,29.004423,7399,199,248238
255,Iasi,2014,23.136876,12747,451,531446
427,Vaslui,2016,21.599622,5709,455,243245
421,Vaslui,2013,20.763906,5397,238,248460
72,Botosani,2018,20.277119,5312,351,244660
73,Botosani,2019,17.310390,4592,397,242340


In [27]:
df[[
    "judet", "an", "rata_migratie_neta_15_64",
    "imigranti_15_64_nr", "emig_15_64_nr", "populatie_15_64"
]].sort_values("rata_migratie_neta_15_64", ascending=True).head(10)

,judet,an,rata_migratie_neta_15_64,imigranti_15_64_nr,emig_15_64_nr,populatie_15_64
131,Caras-Severin,2022,-3.940931,187,796,154532
241,Hunedoara,2022,-3.387230,198,975,229391
406,Timis,2022,-2.775797,644,1835,429066
373,Sibiu,2022,-2.741431,282,965,249140
129,Caras-Severin,2021,-2.732274,136,585,164332
97,Braila,2022,-2.661889,90,558,175815
98,Brasov,2022,-2.557441,498,1377,343703
417,Tulcea,2022,-2.541379,86,396,121981
21,Arad,2022,-2.527456,242,911,264693
372,Sibiu,2021,-2.382694,216,825,255593


In [28]:
# verificare finala + salvare datset

In [29]:
new_vars = [
    "schimbare_ocupati_agri_nr",
    "schimbare_ocupati_agri_pct",
    "schimbare_pondere_ocupati_agri",
    "ocupati_agri_1000ha",
    "migratie_neta_total_nr",
    "rata_migratie_neta_total",
    "migratie_neta_15_64_nr",
    "rata_migratie_neta_15_64"
]

df[new_vars].isna().sum()

schimbare_ocupati_agri_nr         41
schimbare_ocupati_agri_pct        41
schimbare_pondere_ocupati_agri    41
ocupati_agri_1000ha                0
migratie_neta_total_nr             0
rata_migratie_neta_total           0
migratie_neta_15_64_nr             0
rata_migratie_neta_15_64           0
dtype: int64

In [30]:
df[new_vars].describe().T

,count,mean,std,min,25%,50%,75%,max
schimbare_ocupati_agri_nr,410.0,-4028.292683,6626.787124,-37800.000000,-5000.000000,-1700.000000,200.000000,1200.000000
schimbare_ocupati_agri_pct,410.0,-8.526839,14.588266,-51.953125,-12.717880,-3.626087,0.625072,3.921569
schimbare_pondere_ocupati_agri,410.0,-2.039922,3.248365,-15.454547,-2.917847,-0.703512,-0.130539,1.053855
ocupati_agri_1000ha,451.0,494.920793,493.366014,48.089740,171.636472,369.194843,620.253984,3465.120526
migratie_neta_total_nr,451.0,290.942350,1743.518633,-1507.000000,-293.500000,-164.000000,-55.500000,12982.000000
rata_migratie_neta_total,451.0,0.473837,3.624471,-3.064538,-0.770640,-0.413633,-0.175147,30.970113
migratie_neta_15_64_nr,451.0,287.177384,1592.740147,-1191.000000,-244.500000,-135.000000,-37.500000,12296.000000
rata_migratie_neta_15_64,451.0,0.764311,5.217842,-3.940931,-0.961448,-0.531761,-0.200019,43.393908


In [31]:
df.shape

(451, 89)

In [32]:
[col for col in df.columns if "migratie" in col.lower()]

['intensitate_migratie',
 'migratie_neta_total_nr',
 'rata_migratie_neta_total',
 'migratie_neta_15_64_nr',
 'rata_migratie_neta_15_64']

In [33]:
[col for col in df.columns if "ocupati" in col.lower()]

['ocupati_total_nr',
 'ocupati_agri_nr',
 'pondere_ocupati_agri',
 'schimbare_ocupati_agri_nr',
 'schimbare_ocupati_agri_pct',
 'schimbare_pondere_ocupati_agri',
 'ocupati_agri_1000ha']

## Constructed Variable Dictionary

- `schimbare_ocupati_agri_nr` — Annual change in the number of people employed in agriculture, forestry and fishing.
- `schimbare_ocupati_agri_pct` — Annual percentage change in agricultural employment within each county.
- `schimbare_pondere_ocupati_agri` — Annual change, in percentage points, in the share of agricultural employment.
- `ocupati_agri_1000ha` — Agricultural employment per 1,000 hectares of cultivated land.
- `migratie_neta_total_nr` — Total international net migration.
- `rata_migratie_neta_total` — Total international net migration per 1,000 inhabitants.
- `migratie_neta_15_64_nr` — Net international migration among the working-age population (15–64).
- `rata_migratie_neta_15_64` — Working-age net migration per 1,000 people aged 15–64.

In [34]:
df.to_csv(
    "../../data/processed/analysis_dataset.csv",
    index=False,
    encoding="utf-8"
)